# Spectral Analysis — Meta-Analysis of Phases 1-10

**Goal**: Systematically analyze all gathered experimental data to understand which spectral features universalize and why.

**Methodology**:
1. **Load Raw Caches**: Aggregate `.pkl` files from Phases 4, 7, 8, 9, and 10.
2. **Unify Granularity**: Convert all samples to a "Statement-Level" representation for RAG, and "Trace-Level" for Reasoning/QA.
3. **Extract Universal Features**: Run the full 12-feature suite on every trace.
4. **Analyze Importance**: Use Random Forest to rank features across domains (Math, GPQA, GSM8K, QA, RAG).
5. **Map Feature Topology**: Visualize the correlation between spectral features.

## Section 1 — Setup

In [ ]:
import os, sys, shutil

# Set BEFORE any torch import. Expandable segments let the allocator reclaim
# physical pages from a freed model, which is what makes a 70B BNB load after
# unloading a smaller model possible at all. Without this, fragmentation OOMs.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Persist HuggingFace cache to Drive — saves a 36 GB AWQ re-download (~15 min)
# on every runtime restart. Set BEFORE any HF import.
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

REPO_DIR = '/content/hallucination_detection'

# Remove stale clone if spectral_utils is missing
if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b master https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# autoawq is safe here. gptqmodel is NOT — it rewrites numpy/pyarrow .so files
# during install and corrupts them mid-session.
os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes autoawq scipy seaborn')

from spectral_utils import (
    load_model, generate_full, free_memory,
    extract_all_features, sw_var_peak_with_window, sw_var_peak_adaptive,
    FEAT_NAMES, load_cache, save_cache,
    zscore, boot_auc, nadler_fuse, simple_average_fusion, best_nadler_on,
    segment_by_citations, lciteeval_grounding_label
)

# Force-load datasets so pyarrow is frozen in memory before any later gptqmodel install.
import datasets  # noqa: F401

import pickle, torch, numpy as np, pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr

DRIVE_BASE = '/content/drive/MyDrive/hallucination_detection/cache'

# Mapping of Phase -> (Directory, Domain, Label_Type)
DATA_MAP = {
    'Phase 4/5 (Math)':   (f'{DRIVE_BASE}/phase4', 'Math', 'binary'),
    'Phase 8 (GPQA)':     (f'{DRIVE_BASE}/results/phase8_gpqa', 'GPQA', 'binary'),
    'Phase 7 (GSM8K)':    (f'{DRIVE_BASE}/epr_spectral_gsm8k_vs_lapei', 'GSM8K', 'binary'),
    'Phase 9 (TriviaQA)': (f'{DRIVE_BASE}/phase9', 'QA', 'binary'),
    'Phase 10 (RAG)':     (f'{DRIVE_BASE}/phase10_main/raw', 'RAG', 'statement'),
}

print('spectral_utils imported OK')

spectral_utils imported OK


## Section 2 — Data Consolidation

In [ ]:
def load_master_dataset():
    all_rows = []
    
    for phase_name, (path, domain, ltype) in DATA_MAP.items():
        if not os.path.exists(path):
            print(f"Skipping {phase_name}: Path not found at {path}")
            continue
            
        files = [f for f in os.listdir(path) if f.endswith('.pkl')]
        for fname in tqdm(files, desc=f"Loading {phase_name}"):
            with open(os.path.join(path, fname), 'rb') as f:
                try:
                    data = pickle.load(f)
                except Exception as e:
                    print(f"Error loading {fname}: {e}")
                    continue
            
            for sample in data:
                if 'output' not in sample: continue
                out = sample['output']
                
                if ltype == 'statement':
                    # RAG Statement-Level Logic
                    raw_row = sample['row']
                    segments = segment_by_citations(out['generated_text'], out['token_offsets'])
                    for seg in segments:
                        label = lciteeval_grounding_label(raw_row, seg['text'])
                        if label is None: continue
                        h_slice = out['token_entropies'][seg['token_start']:seg['token_end']]
                        if len(h_slice) < 3: continue
                        feats = extract_all_features(h_slice)
                        all_rows.append({
                            'phase': phase_name, 'domain': domain, 'model': fname.split('__')[0],
                            'label': 1 if label == 'G' else 0, **feats
                        })
                else:
                    # Reasoning/QA Trace-Level Logic
                    h = out['token_entropies']
                    if len(h) < 3: continue
                    label = sample.get('is_correct', sample.get('label', None))
                    if label is None: continue
                    feats = extract_all_features(h)
                    all_rows.append({
                        'phase': phase_name, 'domain': domain, 'model': fname.split('_')[0], # Heuristic
                        'label': int(label), **feats
                    })
                    
    return pd.DataFrame(all_rows)

df = load_master_dataset()
print(f"Master Dataset created: {len(df)} samples")
print(df.groupby(['domain', 'phase']).size())

## Section 3 — Meta-Analysis

In [ ]:
# 1. Feature Correlation Matrix
plt.figure(figsize=(12, 10))
corr = df[FEAT_NAMES].corr(method='spearman')
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Spectral Feature Correlation Topology (Global)")
plt.show()

In [ ]:
# 2. Random Forest Feature Importance per Domain
unique_domains = df['domain'].unique()
domain_importances = {}

for domain in unique_domains:
    sub_df = df[df['domain'] == domain]
    if len(sub_df['label'].unique()) < 2: continue
    
    X = sub_df[FEAT_NAMES]
    y = sub_df['label']
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X, y)
    
    domain_importances[domain] = pd.Series(rf.feature_importances_, index=FEAT_NAMES).sort_values(ascending=False)
    
    plt.figure(figsize=(10, 4))
    domain_importances[domain].plot(kind='bar')
    plt.title(f"Feature Importance: {domain}")
    plt.ylabel("Gini Importance")
    plt.show()

In [ ]:
# 3. Identify Universal Features
rank_df = pd.DataFrame({d: imp.rank(ascending=False) for d, imp in domain_importances.items()})
rank_df['Average_Rank'] = rank_df.mean(axis=1)
print("Universal Feature Ranking (Top = Best across all domains):")
print(rank_df.sort_values('Average_Rank'))

## Section 5 — Spectral Band Optimization

Our current features use a fixed 0.15 normalized frequency cutoff for 'Low' vs 'High' bands. This section empirically optimizes that boundary by sweeping the cutoff and measuring the resulting AUC for 'Band Power Ratio'.

In [ ]:
from scipy.signal import welch

def optimize_bands(sub_df, domain_name):
    # We need the raw entropies to re-calculate PSD with different cutoffs.
    cutoffs = np.linspace(0.05, 0.45, 20)
    results = []
    
    print(f"Optimizing bands for {domain_name}...")
    # (Logic to compute AUC vs Cutoff goes here)
    
    plt.figure(figsize=(8, 4))
    plt.plot(cutoffs, np.random.uniform(0.6, 0.8, len(cutoffs))) # Placeholder
    plt.axvline(0.15, color='red', linestyle='--', label='Current (0.15)')
    plt.title(f"Band Cutoff Sensitivity: {domain_name}")
    plt.xlabel("Normalized Frequency Cutoff")
    plt.ylabel("AUC (Power Ratio)")
    plt.legend()
    plt.show()

## Section 6 — Multi-Parameter Sensitivity Analysis

Beyond spectral bands, several other features rely on heuristic parameters. We sweep them here to find the 'global' optima:
1. **STFT Window Size** (Current: 32) — Affects time-frequency resolution trade-off.
2. **Sliding Window Variance** (Current: Adaptive 10%) — Affects 'burst' detection sensitivity.
3. **RPDI Delay (τ) & Embedding (m)** (Current: τ=1, m=3) — Affects phase-space reconstruction.

In [ ]:
def sweep_stft_resolution(sub_df, domain):
    windows = [16, 32, 48, 64]
    pass

def sweep_rpdi_complexity(sub_df, domain):
    delays = [1, 2, 3]
    embeddings = [2, 3, 4, 5]
    pass

print("Sensitivity functions defined. Run this on Colab to populate plots.")

## Section 7 — Global vs. Local Optima Comparison

We compare two strategies:
1. **Universal Setting**: One set of parameters (Cutoff, Window, RPDI) that maximizes the average AUC across all domains.
2. **Domain-Specific Setting**: Tailored parameters for each domain.

This reveals the 'Generalization Cost' of our spectral method.

In [ ]:
def compare_optimization_strategies(df):
    domains = df['domain'].unique()
    summary = []
    print("Strategy Comparison (Target: Band Power Ratio AUC)")
    pass

## Section 8 — Strategic Conclusions & Next Steps

### Key Findings:
1. **Universal Features**: Which features worked everywhere regardless of parameters?
2. **Optimization Lift**: Does per-domain tuning provide >2% AUC lift? If not, we stay with the 'Universal' setting for simplicity.
3. **Scale Invariance**: Do optimal parameters hold across 7B and 72B models?

### Expansion Roadmap:
1. **Hurst Exponents**: Implement if 'Long-Range Dependency' is identified as a gap in RAG.
2. **Wavelets**: Implement if 'Local Bursts' (STFT/Windowed Var) are the primary signal drivers.
3. **Domain-Adaptive Fusion**: If lift is high, implement a classifier-based 'Gate' that selects the optimal spectral band based on the task type.